# Module 8 - Class 2: Your First MLflow Experiment

In [ ]:
# Install MLflow and prepare the breast-cancer data.
!pip -q install mlflow
import mlflow
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42, stratify=data.target)
mlflow.set_experiment('module_8_class_2_experiments')

In [ ]:
# Train and log Logistic Regression plus two Random Forest sizes.
candidates = [('LogisticRegression', LogisticRegression(max_iter=2000), {}), ('RandomForest_50', RandomForestClassifier(n_estimators=50, random_state=42), {'n_estimators':50}), ('RandomForest_200', RandomForestClassifier(n_estimators=200, random_state=42), {'n_estimators':200})]
for name, model, params in candidates:
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        mlflow.log_param('model_type', name)
        mlflow.log_params(params)
        mlflow.log_metric('accuracy', accuracy_score(y_test, predictions))
        mlflow.log_metric('f1_score', f1_score(y_test, predictions))

In [ ]:
# Compare all logged runs by F1 score.
runs = mlflow.search_runs(experiment_names=['module_8_class_2_experiments'])
display(runs[['tags.mlflow.runName', 'metrics.accuracy', 'metrics.f1_score', 'params.n_estimators']].sort_values('metrics.f1_score', ascending=False))
print('Choose the top F1 model only if its gain justifies the additional compute.')